In [35]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage,SystemMessage,BaseMessage
import os
from dotenv import load_dotenv
from langgraph.graph.message import add_messages

In [36]:
load_dotenv()

True

In [37]:
api_key = os.getenv("groq_api")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [38]:
class chatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [39]:
def ChatBot(state: chatState):
    messages = state["messages"]

    response = llm.invoke(messages)

    return {"messages": [response]}

In [40]:
graph = StateGraph(chatState)
graph.add_node('ChatBot',ChatBot)

graph.add_edge(START,'ChatBot')
graph.add_edge('ChatBot',END)

In [41]:
workflow = graph.compile()
initial_state = {"messages": [HumanMessage(content='What is the captial of Pakistan')]}
workflow.invoke(initial_state)

{'messages': [HumanMessage(content='What is the captial of Pakistan', additional_kwargs={}, response_metadata={}, id='c8991590-16c0-46ac-8a41-247a7bb1bed6'),
  AIMessage(content='The capital of Pakistan is **Islamabad**.', additional_kwargs={'reasoning_content': 'The user asks: "What is the captial of Pakistan". They spelled "captial" incorrectly. They want the capital of Pakistan. The answer: Islamabad. Provide correct spelling. Also maybe mention that it\'s the capital city. Provide a brief answer.'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 78, 'total_tokens': 149, 'completion_time': 0.074344267, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.003742606, 'prompt_tokens_details': None, 'queue_time': 0.245009826, 'total_time': 0.078086873}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_bb6a356524', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01